# Week 7 Practice: Data Visualization (Matplotlib and Seaborn)

Practice notebook based on your Week 7 lesson: saving and loading dataframes, exploring a long-form dataset, plotting with Matplotlib, plotting with Seaborn, figure aesthetics, multi-panel figures, and saving figures.

**How to use it**
- Run the setup cells first, then work through the parts in order. Write your answers in the empty code cells.
- Where it says **predict**, write your guess as a comment *before* running the code.
- `<details>` blocks contain hints. Open them only after you've tried.
- Use `?` after a function name (e.g. `sns.lineplot?`) to read the documentation. That is part of the practice.

The dataset is **synthetic** but has the same columns as the `SST_data.csv` from class, so everything you write here will also work on the real file.

## Setup

In [53]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

### Make the dataset
This cell builds a fake two-photon imaging dataset: 15 SST interneurons, 60 image-presentation trials, and 46 timepoints per trial. You don't need to understand the code, but have a look at the structure it creates. Novel images evoke larger responses than familiar ones, and omitted-image trials have a small, later response.

*(If you'd rather use the real class file, skip this cell and run `data = pd.read_csv('SST_data.csv')` instead.)*

In [54]:
rng = np.random.default_rng(42)

n_cells, n_trials = 15, 60
time = np.linspace(-0.5, 1.0, 46)                      # seconds from image presentation
cell_ids = rng.choice(np.arange(1086500000, 1086500100), n_cells, replace=False)
trial_ids = 600 + np.arange(n_trials)

exposure = rng.permutation(["familiar"] * 30 + ["novel"] * 30)
omitted = rng.permutation([True] * 10 + [False] * 50)
pupil = np.clip(rng.normal(np.where(exposure == "novel", 1600, 1400), 250), 300, None)

gain = rng.lognormal(0, 0.3, n_cells)                  # each cell has its own gain
amp = np.where(exposure == "novel", 0.45, 0.25) * np.where(omitted, 0.4, 1.0)
tau = np.where(omitted, 0.30, 0.15)                    # omission response is slower

t = time[None, None, :]
shape = np.where(t > 0, (t / tau[None, :, None]) * np.exp(1 - t / tau[None, :, None]), 0)
trace = gain[:, None, None] * amp[None, :, None] * shape + rng.normal(0, 0.05, (n_cells, n_trials, len(time)))
mean_resp = trace[:, :, (time >= 0) & (time <= 0.5)].mean(axis=2)

idx = pd.MultiIndex.from_product([cell_ids, trial_ids, time], names=["cell_id", "trial_id", "time_from_stim"])
data = idx.to_frame(index=False)
data["dF/F"] = trace.ravel()
data["exposure"] = np.tile(np.repeat(exposure, len(time)), n_cells)
data["omitted"] = np.tile(np.repeat(omitted, len(time)), n_cells)
data["pupil_area"] = np.tile(np.repeat(pupil, len(time)), n_cells)
data["mean_response"] = np.repeat(mean_resp.ravel(), len(time))
data = data[["dF/F", "time_from_stim", "cell_id", "exposure", "trial_id", "omitted", "pupil_area", "mean_response"]]

data

,dF/F,time_from_stim,cell_id,exposure,trial_id,omitted,pupil_area,mean_response
0,0.022304,-0.500000,1086500007,familiar,600,False,1069.325097,0.175038
1,-0.022749,-0.466667,1086500007,familiar,600,False,1069.325097,0.175038
2,-0.061280,-0.433333,1086500007,familiar,600,False,1069.325097,0.175038
3,-0.063897,-0.400000,1086500007,familiar,600,False,1069.325097,0.175038
4,0.008629,-0.366667,1086500007,familiar,600,False,1069.325097,0.175038
...,...,...,...,...,...,...,...,...
41395,0.023722,0.866667,1086500094,familiar,659,False,1841.982478,0.119050
41396,0.072002,0.900000,1086500094,familiar,659,False,1841.982478,0.119050
41397,0.022112,0.933333,1086500094,familiar,659,False,1841.982478,0.119050
41398,-0.040317,0.966667,1086500094,familiar,659,False,1841.982478,0.119050


**Documentation of the columns**

`dF/F` calcium imaging signal · `time_from_stim` time of each row relative to image presentation · `cell_id` the neuron · `exposure` familiar or novel image · `trial_id` each image presentation is one trial · `omitted` whether the image was omitted · `pupil_area` pupil size on that trial · `mean_response` average dF/F over the 500 ms after the image

Each row is **one cell, on one trial, at one timepoint** (long-form format).

---
# Part 1: Saving and loading dataframes

### 1.1 CSV round trip
Here is a small dataframe.

In [55]:
df_mice = pd.DataFrame({
    "mouse": ["M1", "M2", "M3", "M4"],
    "genotype": pd.Categorical(["SST-Cre", "SST-Cre", "WT", "WT"]),
    "session": [1, 2, 1, 3],
    "good_session": [True, False, True, True],
})
df_mice.dtypes

mouse             object
genotype        category
session            int64
good_session        bool
dtype: object

1. Save `df_mice` to `mice.csv` **without** the index, then load it back into `df_mice_loaded`.
2. Compare `df_mice.dtypes` and `df_mice_loaded.dtypes`. Which type changed? Why do you think a CSV can't keep it?
3. Now save it **with** the index (the default) and load it back. What extra column do you get, and how do you avoid it?

In [56]:
# v. Save `df_mice` to `mice.csv` **without** the index, then load it back into `df_mice_loaded`.
df_mice.to_csv("mice.csv")
df_mice_loaded = pd.read_csv("mice.csv")

In [57]:
# 2. Compare `df_mice.dtypes` and `df_mice_loaded.dtypes`. 
# Which type changed? Why do you think a CSV can't keep it?
df_mice.dtypes

mouse             object
genotype        category
session            int64
good_session        bool
dtype: object

In [58]:
df_mice_loaded.dtypes

Unnamed: 0       int64
mouse           object
genotype        object
session          int64
good_session      bool
dtype: object

In [59]:
# v. Save `df_mice` to `mice.csv` **without** the index, then load it back into `df_mice_loaded`.
df_mice.to_csv("mice.csv", index=False)
df_mice_loaded = pd.read_csv("mice.csv")

In [60]:
df_mice.dtypes

mouse             object
genotype        category
session            int64
good_session        bool
dtype: object

In [61]:
df_mice_loaded.dtypes

mouse           object
genotype        object
session          int64
good_session      bool
dtype: object

### 1.2 Another format
CSV doesn't store types. Try `df_mice.to_pickle('mice.pkl')` and `pd.read_pickle('mice.pkl')`. Are the dtypes preserved this time? (Pickle files are only safe to share with people you trust, and they can break between pandas versions. Feather and hdf5 are the sturdier choices.)

---
# Part 2: Exploring the dataset

### 2.1 First look
1. What is the shape of `data`? How many rows? (**predict** first: 15 cells × 60 trials × 46 timepoints = ?)
2. List the columns and their dtypes.
3. How many different cells are there? How many different trials?

In [62]:
# data
# data.head()
data.columns
data.shape
data.describe()

,dF/F,time_from_stim,cell_id,trial_id,pupil_area,mean_response
count,41400.000000,41400.000000,4.140000e+04,41400.000000,41400.000000,41400.000000
mean,0.089553,0.250000,1.086500e+09,629.500000,1470.715701,0.214086
std,0.136644,0.442536,2.704465e+01,17.318311,270.712624,0.103086
min,-0.194390,-0.500000,1.086500e+09,600.000000,866.988430,0.041668
25%,-0.000778,-0.133333,1.086500e+09,614.750000,1302.743815,0.135648
50%,0.052235,0.250000,1.086500e+09,629.500000,1453.230841,0.202122
75%,0.144127,0.633333,1.086500e+09,644.250000,1612.065870,0.267041
max,0.947550,1.000000,1.086500e+09,659.000000,2328.465617,0.585793


In [63]:
# 3. How many different cells are there? How many different trials?
data['cell_id'].nunique()



15

In [64]:
data["trial_id"].nunique()

60

### 2.2 Counting trials by condition
How many **familiar** and how many **novel** trials are there? 

(Careful: each trial has many rows, so `data['exposure'].value_counts()` counts rows, not trials.)

<details><summary>Hint</summary>Keep only one row per trial first, e.g. with <code>drop_duplicates('trial_id')</code>, then count.</details>

In [65]:
# How many **familiar** and how many **novel** trials are there? 
familiar_trials = data[data['exposure'] == 'familiar'].shape[0]
novel_trials = data[data['exposure'] == 'novel'].shape[0]

In [66]:
data["exposure"]. value_counts()

exposure
familiar    20700
novel       20700
Name: count, dtype: int64

In [67]:
data_unique = data.drop_duplicates(subset=["trial_id"])
data_unique["exposure"].value_counts()



exposure
familiar    30
novel       30
Name: count, dtype: int64

### 2.3 One cell, one trial
1. Create `single_trial_data` containing only trial 605 and the **first** cell in `data.cell_id.unique()`.
2. How many rows does it have? Does that match the number of timepoints?
3. Is trial 605 familiar or novel? Was the image omitted?

In [77]:
data.columns
data["trial_id"].count()
len(data["trial_id"])
(data["trial_id"] == 605).sum()

np.int64(690)

In [78]:
first_cell = data.cell_id.unique()[0]
single_trial_data = data.loc[
    (data["trial_id"] == 605) &
    (data["cell_id"] == first_cell)
].copy()

In [79]:
single_trial_data

,dF/F,time_from_stim,cell_id,exposure,trial_id,omitted,pupil_area,mean_response
230,0.009839,-0.500000,1086500007,novel,605,False,1924.807074,0.307465
231,0.041026,-0.466667,1086500007,novel,605,False,1924.807074,0.307465
232,-0.019687,-0.433333,1086500007,novel,605,False,1924.807074,0.307465
233,0.026058,-0.400000,1086500007,novel,605,False,1924.807074,0.307465
234,-0.013292,-0.366667,1086500007,novel,605,False,1924.807074,0.307465
235,-0.005877,-0.333333,1086500007,novel,605,False,1924.807074,0.307465
236,0.041476,-0.300000,1086500007,novel,605,False,1924.807074,0.307465
237,-0.099653,-0.266667,1086500007,novel,605,False,1924.807074,0.307465
238,-0.064824,-0.233333,1086500007,novel,605,False,1924.807074,0.307465
239,-0.074109,-0.200000,1086500007,novel,605,False,1924.807074,0.307465


In [ ]:
# 2. How many rows does it have? Does that match the number of timepoints?
single_trial_data.shape[0]

0

### 2.4 Filtering with several conditions
How many rows have **all** of these: `omitted` is `True`, `time_from_stim` is greater than 0, and `dF/F` is greater than 0.2?

<details><summary>Hint</summary>Combine boolean conditions with <code>&</code>, and put each condition in parentheses.</details>

---
# Part 3: Plotting a single trace with Matplotlib

### 3.1 A first plot
1. Pull out the `dF/F` values and the `time_from_stim` values of `single_trial_data` as numpy arrays.
2. Plot the trace against its **index** (no time axis). Then plot it against **time**.

### 3.2 Labels and size
Make the time-axis plot presentable: x label, y label (with units, think about what `dF/F` means), a title that includes the trial number (use an f-string), and a figure size of 5 by 3.

### 3.3 Mark the stimulus
Add a vertical dashed line at time 0 so it is obvious when the image appears. (Look up `plt.axvline`.)

### 3.4 Two traces on one plot
For the same first cell, pick one **familiar** trial and one **novel** trial (non-omitted). Plot both on the same axes, give each a `label=`, and add a legend with `plt.legend()`. Does the novel trial look larger?

<details><summary>Hint</summary>Filter <code>data</code> for the cell, the exposure, and <code>omitted == False</code>, then take the first <code>trial_id</code> you find.</details>

---
# Part 4: Seaborn basics

### 4.1 Average across everything
1. Plot `dF/F` against `time_from_stim` using `sns.lineplot` on the whole dataset.
2. **Predict:** what is the line, and what is the shaded band? Check the documentation (`sns.lineplot?`) to see if you were right.

### 4.2 Familiar vs novel
1. Make a barplot of `dF/F` by `exposure`.
2. Make a lineplot of `dF/F` over time, split by `exposure` using `hue`.
3. Why do these two plots tell different stories? Which is more informative?

### 4.3 Reading the documentation
Using `sns.lineplot?` (or the Seaborn website), figure out how to:
- (a) turn the shaded band **off**
- (b) show the standard error instead of the confidence interval
- (c) use a different **line style** for omitted vs non-omitted trials (`style=`)

### 4.4 Histograms
Use this smaller sample so the plots are quick.

In [ ]:
data_sample = data.sample(1000, random_state=0)

1. Plot a histogram of `pupil_area`.
2. Split it by `exposure`.
3. Because there are different numbers of points in each group, the default can mislead. Look up `stat` and `common_norm` in `sns.histplot?` and make the groups comparable.
4. **Challenge:** add a smooth density curve (`kde`), change the number of bins, and add a title.

### 4.5 Two panels by hand
Make a figure with two side-by-side axes (`fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)`). Put the familiar pupil-area histogram on the left and the novel one on the right.

Seaborn functions accept an `ax=` argument, so you can write `sns.histplot(..., ax=axes[0])`.

---
# Part 5: Figure aesthetics

### 5.1 Styles and contexts
Using `data_sample`, draw a strip plot of `mean_response` by `exposure` with `sns.catplot`. Do it three times with different style / context combinations:
- `'whitegrid'` and `'talk'`
- `'white'` and `'poster'`
- `'ticks'` and `'notebook'`

What changes? Which would you use in a paper, in a slide, and on a poster?

### 5.2 Point size and colors
1. Make the points smaller (look up the size argument).
2. Colour the points by `exposure` using `hue='exposure'` and a **colorblind-friendly** palette (`palette='colorblind'`). Try another palette from `sns.color_palette?` and see how it looks.
3. Set a single colour for all points instead (`color=`).

### 5.3 Labels, title, despine
Take the lineplot split by `exposure` from 4.2 and:
- (a) rename the axes so that they say something meaningful
- (b) add a title
- (c) remove the top and right borders with `sns.despine()`
- (d) increase the font size using `sns.set_context('notebook', font_scale=1.3)`

### 5.4 Challenges
- (a) Give the legend a visible frame with a black edge. (Hint: `plt.legend(...)` or `sns.move_legend`.)
- (b) Change the font of the whole figure. (Hint: `plt.rcParams`.)

---
# Part 6: Multiple plots and trial-level data

### 6.1 A grid of plots
Use `sns.relplot` with `kind='line'` to plot `dF/F` over time, with one column per `exposure` and one row per `omitted`. Then write one sentence in a markdown cell describing what differs between familiar and novel, and between omitted and not omitted trials.

### 6.2 A trap: repeated values
`mean_response` is repeated on every timepoint of a trial, so plotting it directly treats one number as 46 separate measurements.

1. Build a **trial-level** dataframe `trial_df` with exactly one row per cell and trial. **Predict** its length first (hint: 15 × 60), and check it.
2. Make a box plot (`sns.catplot(kind='box', ...)`) of `mean_response` by `exposure`, with one panel per `omitted` value (`col=`).
3. Compare it with the same plot made on the full `data`. Does the picture change? Do the error bars or box widths change?

<details><summary>Hint</summary><code>data.drop_duplicates(['cell_id', 'trial_id'])</code> keeps one row per cell and trial.</details>

---
# Part 7: Saving figures

### 7.1 Resolution and cropping
1. Make a lineplot of `dF/F` over time with `hue='exposure'`, figsize (8, 4), with axis labels and a title. Save it as `fig_dpi100.png` with `dpi=100`, and again as `fig_dpi300.png` with `dpi=300`. Use `bbox_inches='tight'` so nothing is cut off.
2. Load both back with `plt.imread` and print their `.shape`. Why is the second one bigger? **Predict** the shapes first (figure size × dpi).
3. Save one more copy as a vector file (`.svg` or `.pdf`). When would you prefer it?

---
# Part 8: Mini-project, a figure for a paper

Build a single figure with **two panels** side by side (`fig, axes = plt.subplots(1, 2, figsize=(11, 4))`):

- **Panel A:** mean dF/F over time, familiar vs novel, **non-omitted trials only**, with a dashed vertical line at stimulus onset.
- **Panel B:** mean response (0 to 500 ms) for familiar vs novel, using the trial-level data averaged per **cell** first (one point per cell and condition, so n = 15). Show the individual cells as points and the mean as a bar or point.

Checklist: a colorblind palette used consistently in both panels, meaningful axis labels, no legend duplicated in both panels, top and right spines removed, panel letters "A" and "B", and saved as `figure1.png` at 300 dpi with nothing cut off.

<details><summary>Hint for panel B</summary>Use <code>trial_df.groupby(['cell_id', 'exposure'])['mean_response'].mean().reset_index()</code> to get one value per cell and condition.</details>

---
## Reflection
In a markdown cell, answer in your own words:
1. What does "long-form" data mean, and why does it make Seaborn easier to use?
2. What is the shaded band in `sns.lineplot` by default?
3. Why can plotting repeated values (like `mean_response`) mislead?
4. When would you pick `hue`, and when would you pick `col`/`row`?